# Documentação:

1. Justificativa da Imputação pela Mediana (Tarefas 2 e Documentação)
Uso da Mediana vs. Média:
Optou-se pelo uso da Mediana para imputação dos valores ausentes na coluna valor porque o conjunto de dados possui outliers extremos (ex.: despesas de R$ 12.500, R$ 8.900 e R$ 45.000). A média matemática é altamente sensível a valores discrepantes, o que distorceria a estimativa para cima. A mediana representa o valor central real da distribuição (50º percentil), sendo uma medida estatística robusta e imune a distorções causadas por anomalias.

2. Justificativa da Sinalização de Outliers (Tarefa 3 e Visão de Auditoria)
Sinalização (is_outlier) vs. Exclusão de Registros:
Em arquiteturas de dados voltadas para Fintechs e Auditoria, o descarte prematuro de registros atípicos compromete o histórico de transações e a detecção de potenciais fraudes. Ao criar a flag booleana is_outlier, mantemos a rastreabilidade total da base original (Data Provenance), permitindo que relatórios gerenciais e modelos estatísticos filtrem transações sem destruir o registro original do banco de dados.

# Simulação com 50 Registros

In [22]:
import sqlite3
import pandas as pd
import numpy as np

# ==============================================================================
# SIMULAÇÃO: Criar tabelas 'despesas' e 'estudantes' com 50 registros (brutos)
# ==============================================================================
def popular_banco_inicial(nome_banco="fintech_integrada.db"):
    np.random.seed(42)

    # 1. 50 Registros de Despesas (com erros de datas, nulos, categorias e outliers)
    ids_estudante = [f"EST_{i:03d}" for i in range(1, 46)] + ["EST_001", "EST_002", None, "EST_010", "EST_015"]
    datas = ["2026-09-01", "01/09/2026", "2026/09/02", "INVALID_DATE", None] * 10
    categorias = ["Alimentacao", " ALIMENTACAO ", "transporte", None, "Lazer"] * 10
    valores = np.random.normal(loc=150, scale=30, size=50).tolist()

    # Inserindo anomalias e outliers
    valores[5] = None
    valores[12] = -50.0
    valores[25] = 12500.0  # Outlier
    valores[38] = 8900.0   # Outlier

    df_despesas = pd.DataFrame({
        'id_estudante': ids_estudante,
        'data': datas,
        'categoria': categorias,
        'valor': valores
    })

    # 2. Tabela de estudantes
    df_estudantes = pd.DataFrame({
        'id_estudante': [f"EST_{i:03d}" for i in range(1, 46)],
        'nome': [f"Estudante {i}" for i in range(1, 46)]
    })

    # 3. Gravando ambas as tabelas no Banco SQLite de uma só vez
    conn = sqlite3.connect(nome_banco)
    df_despesas.to_sql("despesas", conn, if_exists='replace', index=False)
    df_estudantes.to_sql("estudantes", conn, if_exists='replace', index=False)
    conn.close()

    print(f"✅ Tabelas 'despesas' e 'estudantes' salvas com sucesso em '{nome_banco}'.")

if __name__ == "__main__":
    popular_banco_inicial()

✅ Tabelas 'despesas' e 'estudantes' salvas com sucesso em 'fintech_integrada.db'.


# Código e tratativas

In [23]:
# ==============================================================================
# TAREFA 1: Ingestão e Inspeção
# ==============================================================================
def tarefa_1_ingestao_e_inspecao(nome_banco="fintech_integrada.db"):
    print("="*60)
    print("TAREFA 1: INGESTÃO E INSPEÇÃO INICIAL")
    print("="*60)

    conn = sqlite3.connect(nome_banco)
    df_despesas = pd.read_sql("SELECT * FROM despesas", conn)
    df_estudantes = pd.read_sql("SELECT * FROM estudantes", conn)
    conn.close()

    print("\n--- Relatório de Diagnóstico Inicial (Despesas) ---")
    print("\n1. Contagem de Valores Nulos por Coluna:")
    print(df_despesas.isnull().sum())

    print("\n2. Tipos de Dados:")
    print(df_despesas.dtypes)

    duplicados = df_despesas.duplicated().sum()
    print(f"\n3. Registros Duplicados Encontrados: {duplicados}")

    return df_despesas, df_estudantes

# ==============================================================================
# TAREFA 2: Limpeza e Imputação
# ==============================================================================
def tarefa_2_limpeza_e_imputacao(df_despesas):
    print("\n" + "="*60)
    print("TAREFA 2: LIMPEZA E IMPUTAÇÃO")
    print("="*60)

    df_clean = df_despesas.copy()

    # 1. Converter coluna data para datetime
    df_clean['data'] = pd.to_datetime(df_clean['data'], format='mixed', errors='coerce')

    # 2. Padronizar categorias e preencher nulos com "nao_informado"
    df_clean['categoria'] = (
        df_clean['categoria']
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({'none': np.nan, 'nan': np.nan})
    )
    df_clean['categoria'] = df_clean['categoria'].fillna("nao_informado")

    # 3. Tratar valores inválidos (<= 0) e imputar a MEDIANA
    df_clean['valor'] = df_clean['valor'].apply(lambda x: np.nan if pd.notnull(x) and x <= 0 else x)
    mediana_valor = df_clean['valor'].median()
    df_clean['valor'] = df_clean['valor'].fillna(mediana_valor)

    # Imputação da Moda para datas inválidas
    moda_data = df_clean['data'].mode()[0]
    df_clean['data'] = df_clean['data'].fillna(moda_data)

    print(f"✅ Conversão de datas concluída.")
    print(f"✅ Categorias padronizadas e nulos preenchidos com 'nao_informado'.")
    print(f"✅ Imputação da Mediana ({mediana_valor:.2f}) realizada para valores ausentes/inválidos.")

    return df_clean

# ==============================================================================
# TAREFA 3: Tratamento de Anomalias (Outliers via IQR)
# ==============================================================================
def tarefa_3_tratamento_anomalias(df_limpo, df_bruto, coluna='valor'):
    print("\n" + "="*60)
    print("TAREFA 3: TRATAMENTO DE ANOMALIAS (IQR)")
    print("="*60)

    df_out = df_limpo.copy()

    # Cálculo do IQR sobre os dados válidos originais (sem contaminação da mediana)
    valores_validos = df_bruto.loc[df_bruto[coluna] > 0, coluna].dropna()
    q1 = valores_validos.quantile(0.25)
    q3 = valores_validos.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - (1.5 * iqr)
    limite_superior = q3 + (1.5 * iqr)

    # Flag booleana is_outlier
    df_out['is_outlier'] = ((df_out[coluna] < limite_inferior) | (df_out[coluna] > limite_superior)).astype(bool)

    total_outliers = df_out['is_outlier'].sum()
    print(f"Limites IQR: Inferior = {limite_inferior:.2f} | Superior = {limite_superior:.2f}")
    print(f"✅ Sinalização concluída: {total_outliers} outlier(s) marcado(s) na coluna 'is_outlier'.")

    return df_out

# ==============================================================================
# TAREFA 4: Validação Final, Métrica e Persistência
# ==============================================================================
def tarefa_4_validacao_e_relatorio(df_bruto, df_final, nome_banco="fintech_integrada.db"):
    print("\n" + "="*60)
    print("TAREFA 4: VALIDAÇÃO FINAL E RELATÓRIO DE QUALIDADE")
    print("="*60)

    # Deduplicação Final
    df_final = df_final.drop_duplicates().copy()

    # Cálculo da Métrica de Completitude
    completitude_antes = (df_bruto.notnull().sum().sum() / df_bruto.size) * 100
    completitude_depois = (df_final.notnull().sum().sum() / df_final.size) * 100

    print("\n--- Comparativo Antes vs. Depois ---")
    print(f"Total de Registros Brutos: {len(df_bruto)} | Registros Finais Limpos: {len(df_final)}")
    print(f"Métrica de Completitude (Antes) : {completitude_antes:.2f}%")
    print(f"Métrica de Completitude (Depois): {completitude_depois:.2f}%")

    print("\nNulos Restantes por Coluna no Dado Limpo:")
    print(df_final.isnull().sum())

    # Formatação da data antes de gravar no SQL
    df_final['data'] = df_final['data'].dt.strftime('%Y-%m-%d')

    # Salvar na tabela despesas_clean
    conn = sqlite3.connect(nome_banco)
    df_final.to_sql("despesas_clean", conn, if_exists='replace', index=False)
    conn.close()

    print(f"\n✅ Base final limpa salva com sucesso na tabela 'despesas_clean' do banco '{nome_banco}'.")

# ==============================================================================
# EXECUÇÃO DO PIPELINE COMPLETO
# ==============================================================================
if __name__ == "__main__":
    popular_banco_inicial()

    # Execução encadeada exata dos requisitos
    df_bruto, df_estudantes = tarefa_1_ingestao_e_inspecao()
    df_passo2 = tarefa_2_limpeza_e_imputacao(df_bruto)
    df_passo3 = tarefa_3_tratamento_anomalias(df_passo2, df_bruto)
    tarefa_4_validacao_e_relatorio(df_bruto, df_passo3)

✅ Tabelas 'despesas' e 'estudantes' salvas com sucesso em 'fintech_integrada.db'.
TAREFA 1: INGESTÃO E INSPEÇÃO INICIAL

--- Relatório de Diagnóstico Inicial (Despesas) ---

1. Contagem de Valores Nulos por Coluna:
id_estudante     1
data            10
categoria       10
valor            1
dtype: int64

2. Tipos de Dados:
id_estudante     object
data             object
categoria        object
valor           float64
dtype: object

3. Registros Duplicados Encontrados: 0

TAREFA 2: LIMPEZA E IMPUTAÇÃO
✅ Conversão de datas concluída.
✅ Categorias padronizadas e nulos preenchidos com 'nao_informado'.
✅ Imputação da Mediana (143.10) realizada para valores ausentes/inválidos.

TAREFA 3: TRATAMENTO DE ANOMALIAS (IQR)
Limites IQR: Inferior = 69.62 | Superior = 222.62
✅ Sinalização concluída: 2 outlier(s) marcado(s) na coluna 'is_outlier'.

TAREFA 4: VALIDAÇÃO FINAL E RELATÓRIO DE QUALIDADE

--- Comparativo Antes vs. Depois ---
Total de Registros Brutos: 50 | Registros Finais Limpos: 50
Métrica